In [11]:
# ================================
# 📦 IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np

from sklearn.neighbors import NearestNeighbors
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

pd.set_option('display.max_colwidth', None)

# ================================
# 📂 LOAD DATA
# ================================
df = pd.read_csv("../data/processed/clean_data.csv")

products = pd.read_csv("../data/raw/amazon-product-review.csv")[['asins', 'name']]
products = products.drop_duplicates()
products.columns = ['item_id', 'product_name']

df = df.merge(products, on='item_id', how='left')

# Clean product names (UI improvement)
df['product_name'] = df['product_name'].fillna("Unknown Product")
df['product_name'] = df['product_name'].str.slice(0, 60)

# Fast lookup dictionary (IMPORTANT OPTIMIZATION)
item_name_map = dict(zip(df['item_id'], df['product_name']))

print("Data shape:", df.shape)
print(df.head())

# ================================
# 🥇 1. POPULARITY MODEL (WEIGHTED)
# ================================
popularity_df = df.groupby(['item_id', 'product_name']).agg({
    'rating': ['mean', 'count']
})

popularity_df.columns = ['avg_rating', 'rating_count']

# Weighted rating (industry-style)
C = popularity_df['avg_rating'].mean()
m = 5

popularity_df['score'] = (
    (popularity_df['rating_count'] / (popularity_df['rating_count'] + m)) * popularity_df['avg_rating']
    + (m / (popularity_df['rating_count'] + m)) * C
)

popularity_df = popularity_df.sort_values(by='score', ascending=False)

def recommend_popular(n=5):
    return popularity_df.head(n)[['avg_rating', 'rating_count', 'score']]

print("\nTop Popular Items:\n", recommend_popular())

# ================================
# 🤝 2. KNN (USER-BASED)
# ================================
user_item_matrix = df.pivot_table(
    index='user_id',
    columns='item_id',
    values='rating'
).fillna(0)

print("\nUser-Item Matrix Shape:", user_item_matrix.shape)

model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(user_item_matrix.values)

def recommend_knn(user_id, n=5):
    if user_id not in user_item_matrix.index:
        return "User not found"
    
    user_vector = user_item_matrix.loc[user_id].values.reshape(1, -1)
    distances, indices = model_knn.kneighbors(user_vector, n_neighbors=n+1)
    
    return list(user_item_matrix.index[indices.flatten()[1:]])

sample_user = df['user_id'].iloc[0]
print("\nKNN Similar Users:\n", recommend_knn(sample_user))

# ================================
# 🔥 3. SVD MODEL
# ================================
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2)

model_svd = SVD()
model_svd.fit(trainset)

predictions = model_svd.test(testset)
print("\nSVD RMSE:", accuracy.rmse(predictions))

# ================================
# 🎯 SVD RECOMMENDATIONS (OPTIMIZED)
# ================================
def recommend_svd(user_id, n=5):
    if user_id not in df['user_id'].values:
        return "User not found"
    
    user_items = set(df[df['user_id'] == user_id]['item_id'])
    all_items = df['item_id'].unique()
    
    predictions = [
        (item, model_svd.predict(user_id, item).est)
        for item in all_items if item not in user_items
    ]
    
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    result = [
        (item_name_map.get(item, "Unknown"), round(score, 2))
        for item, score in predictions[:n]
    ]
    
    return pd.DataFrame(result, columns=["Product", "Predicted Rating"])

print("\nSVD Recommendations:\n", recommend_svd(sample_user))

# ================================
# 🎯 ITEM-BASED SIMILARITY
# ================================
model_item = NearestNeighbors(metric='cosine', algorithm='brute')
model_item.fit(user_item_matrix.T.values)

def recommend_similar_items(item_id, n=5):
    if item_id not in user_item_matrix.columns:
        return "Item not found"
    
    item_vector = user_item_matrix[item_id].values.reshape(1, -1)
    distances, indices = model_item.kneighbors(item_vector, n_neighbors=n+1)
    
    similar_items = user_item_matrix.columns[indices.flatten()[1:]]
    
    result = [
        item_name_map.get(item, "Unknown")
        for item in similar_items
    ]
    
    return result

sample_item = df['item_id'].iloc[0]
print("\nSimilar Items:\n", recommend_similar_items(sample_item))

Data shape: (1009, 4)
              user_id     item_id  rating       product_name
0          Cristina M  B00QJDU3KY     5.0  Kindle Paperwhite
1               Ricky  B00QJDU3KY     5.0  Kindle Paperwhite
2       Tedd Gardiner  B00QJDU3KY     4.0  Kindle Paperwhite
3              Dougal  B00QJDU3KY     5.0  Kindle Paperwhite
4  Miljan David Tanic  B00QJDU3KY     5.0  Kindle Paperwhite

Top Popular Items:
                                                                          avg_rating  \
item_id    product_name                                                               
B0716JZKLT All-New Amazon Kid-Proof Case for Amazon Fire HD 8 Tablet (7         5.0   
B01A08E70K Kindle E-reader - Black                                              5.0   
B01KIOU214 Amazon Echo - Black                                                  5.0   
B01M4NRFXX All-New Fire HD 8 Kids Edition Tablet                                5.0   
B00LWHUAF0 Fire HD 7 Tablet                                          

In [5]:
import os
print(os.getcwd())

d:\Files\DATA ANALYTICS COURSE\Projects\amazon-recommendation-system\notebooks
